# SKEW Distribution Properties

This notebook validates SKEW as one input to the composite market-risk indicator. It compares raw and log-transformed SKEW Close, checks serial dependence and broad regime stability, and evaluates the exact two-stage rolling transformation used by the composite model.

It does not define a standalone SKEW event, trading entry, or strategy.

> **Live-data notebook.** The saved result tables reproduce the latest validated snapshot dated 2026-07-26. Run all cells to refresh the live data, tables, and charts; Git history preserves earlier snapshots.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
import yfinance.cache as yf_cache
from scipy import stats

from src.market_risk.skew import compute_skew_risk_frame

RUN_TIMESTAMP_UTC = pd.Timestamp.now(tz="UTC")
ROLLING_MONTHS = 120
print(f"Notebook run timestamp (UTC): {RUN_TIMESTAMP_UTC.isoformat()}")


def distribution_stats(series, name):
    clean = series.dropna().astype(float)
    sample = clean.sample(5000, random_state=42) if len(clean) > 5000 else clean
    shapiro_stat, shapiro_p = stats.shapiro(sample)
    jb = stats.jarque_bera(clean)
    return pd.Series({
        "Name": name,
        "Count": len(clean),
        "Mean": clean.mean(),
        "Std": clean.std(),
        "Median": clean.median(),
        "Min": clean.min(),
        "Max": clean.max(),
        "Skewness": stats.skew(clean),
        "Kurtosis_Excess": stats.kurtosis(clean),
        "Lag1_Autocorrelation": clean.autocorr(lag=1),
        "Shapiro_Stat": shapiro_stat,
        "Shapiro_p": shapiro_p,
        "Jarque_Bera": jb.statistic,
        "Jarque_Bera_p": jb.pvalue,
    })

## Data preparation

Daily Close is used for the high-frequency distribution check. Completed-month Close is used for historical monthly diagnostics. The latest daily Close is inserted separately as the provisional current-month input to the live model.

In [ ]:
yf_cache.set_cache_location(tempfile.mkdtemp(prefix="skew-yfinance-"))
end_exclusive = (RUN_TIMESTAMP_UTC.tz_localize(None).normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
raw = yf.download("^SKEW", start="1990-01-01", end=end_exclusive, auto_adjust=False, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

daily_close = raw["Close"].dropna().astype(float).rename("SKEW_Close")
monthly_all = daily_close.resample("ME").last().dropna()
current_month_end = RUN_TIMESTAMP_UTC.tz_localize(None).normalize() + pd.offsets.MonthEnd(0)
completed_monthly = monthly_all.loc[monthly_all.index < current_month_end].copy()
model_monthly = completed_monthly.copy()
model_monthly.loc[current_month_end] = daily_close.iloc[-1]
model_monthly = model_monthly.sort_index()

daily_log = np.log(daily_close).rename("Log_SKEW_Close")
monthly_log = np.log(completed_monthly).rename("Log_Monthly_SKEW_Close")

print(f"Daily sample: {daily_close.index.min().date()} to {daily_close.index.max().date()}")
print(f"Completed-month sample: {completed_monthly.index.min().date()} to {completed_monthly.index.max().date()}")
print(f"Provisional model month: {current_month_end.date()} using {daily_close.index[-1].date()} Close")

Daily sample: 1990-01-02 to 2026-07-24
Completed-month sample: 1990-01-31 to 2026-06-30
Provisional model month: 2026-07-31 using 2026-07-24 Close


## Raw and log distribution comparison

In [ ]:
series_to_plot = [
    (daily_close, "Raw daily SKEW Close"),
    (daily_log, "Log daily SKEW Close"),
    (completed_monthly, "Raw completed-month SKEW Close"),
    (monthly_log, "Log completed-month SKEW Close"),
]
summary = pd.DataFrame([distribution_stats(series, name) for series, name in series_to_plot])
display(summary[["Name", "Count", "Skewness", "Kurtosis_Excess", "Lag1_Autocorrelation", "Shapiro_p", "Jarque_Bera_p"]])

                               Name  Count  Skewness  Kurtosis_Excess  Lag1_Autocorrelation
0              Raw daily SKEW Close   9133     1.284            1.576                 0.963
1              Log daily SKEW Close   9133     1.046            0.772                 0.962
2  Raw completed-month SKEW Close    438     1.285            1.194                 0.853
3  Log completed-month SKEW Close    438     1.079            0.587                 0.851

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (series, title) in zip(axes.flat, series_to_plot):
    clean = series.dropna()
    axis.hist(clean, bins=50, density=True, alpha=0.65)
    x = np.linspace(clean.min(), clean.max(), 300)
    axis.plot(x, stats.norm.pdf(x, clean.mean(), clean.std()), linewidth=2)
    axis.set_title(title)
    axis.set_ylabel("Density")
plt.tight_layout()
plt.show()

### Saved distribution charts

![Raw and log SKEW distribution histograms](../../reports/generated/skew_distribution/distribution_diagnostics.svg)

## Q-Q plots

The plots inspect the shape of the center and tails. Log transformation improves the distribution, but it does not justify treating the full SKEW history as IID normal data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (series, title) in zip(axes.flat, series_to_plot):
    stats.probplot(series.dropna(), dist="norm", plot=axis)
    axis.set_title(f"Q-Q plot: {title}")
plt.tight_layout()
plt.show()

## Regime comparison of completed-month log SKEW Close

In [ ]:
periods = {
    "1990-2007": ("1990-01-01", "2007-12-31"),
    "2008-2019": ("2008-01-01", "2019-12-31"),
    "2020-present": ("2020-01-01", None),
}
regime_rows = []
for name, (start, end) in periods.items():
    subset = monthly_log.loc[start:end]
    if len(subset) >= 10:
        regime_rows.append(distribution_stats(subset, name))
regime_summary = pd.DataFrame(regime_rows)
display(regime_summary[["Name", "Count", "Mean", "Std", "Skewness", "Jarque_Bera_p"]])

fig, axes = plt.subplots(1, len(regime_rows), figsize=(16, 5))
for axis, row in zip(np.atleast_1d(axes), regime_rows):
    name = row["Name"]
    start, end = periods[name]
    subset = monthly_log.loc[start:end].dropna()
    stats.probplot(subset, dist="norm", plot=axis)
    axis.set_title(f"{name}: log monthly Close")
plt.tight_layout()
plt.show()

           Name  Count   Mean    Std  Skewness  Jarque_Bera_p
0     1990-2007    216  4.748  0.041     0.365           0.087
1     2008-2019    144  4.816  0.067     0.475           0.066
2  2020-present     78  4.944  0.091    -0.434           0.246

## Two-stage rolling transformation used by the composite model

The official SKEW parameter is calculated as follows:

1. transform monthly SKEW with the natural logarithm;
2. calculate `(Log SKEW - trailing 120-month Log mean) / trailing 120-month Log mean`;
3. standardize that deviation with its trailing 120-month mean and standard deviation;
4. map the resulting Z-score to 0–1 with the standard normal CDF.

Both rolling windows include the current month. The CDF result is a bounded normal-score mapping, not a crash probability or an empirical historical percentile.

In [ ]:
model = compute_skew_risk_frame(model_monthly.to_frame("SKEW_Close"), window=ROLLING_MONTHS)
model["Normal_Risk_Score_Pct"] = model["Normal_Risk_Score"] * 100
valid_z = model["Z_Score"].dropna()
model_summary = distribution_stats(valid_z, "Composite-model SKEW Z")
display(model_summary[["Name", "Count", "Mean", "Std", "Skewness", "Kurtosis_Excess", "Lag1_Autocorrelation"]])

normal_ks = stats.kstest(valid_z, "norm")
latest_model = model.dropna(subset=["Z_Score"]).iloc[-1]
print(f"Z vs N(0,1) KS p-value: {normal_ks.pvalue:.8g}")
print(f"Latest SKEW: {latest_model['SKEW_Close']:.4f}")
print(f"Latest Z-score: {latest_model['Z_Score']:.4f}")
print(f"Latest bounded normal-score: {latest_model['Normal_Risk_Score_Pct']:.2f}%")

Name                     Composite-model SKEW Z
Count                                       201
Mean                                      0.367
Std                                       1.212
Skewness                                 -0.241
Kurtosis_Excess                          -0.199
Lag1_Autocorrelation                      0.591

Z vs N(0,1) KS p-value: 0.000022
Latest SKEW: 147.2800
Latest Z-score: 0.1310
Latest bounded normal-score: 55.19%


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(model.index, model["Z_Score"], linewidth=1)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].axhline(2, linestyle="--", linewidth=1)
axes[0].axhline(-2, linestyle="--", linewidth=1)
axes[0].set_title("Composite-model SKEW Z-score through time")
axes[0].grid(alpha=0.25)

axes[1].hist(valid_z, bins=30, density=True, alpha=0.65)
x = np.linspace(valid_z.min(), valid_z.max(), 300)
axes[1].plot(x, stats.norm.pdf(x), linewidth=2, label="Standard normal")
axes[1].set_title("Distribution of transformed SKEW Z-score")
axes[1].legend()
plt.tight_layout()
plt.show()

### Saved regime and rolling-model charts

![SKEW regime Q-Q plots and two-stage rolling Z-score](../../reports/generated/skew_distribution/rolling_model_diagnostics.svg)

## Interpretation boundary

Log transformation improves the shape of SKEW but does not make the full history IID normal. The rising level and widening dispersion across regimes support rolling adjustment. After the two-stage transformation, skewness and excess kurtosis are closer to normal-like values, but the series remains serially dependent and is not calibrated as standard normal.

Accordingly, the mapped value is used only as one bounded component of the composite market-risk indicator. Forward returns, drawdowns, event rules, and trading thresholds remain outside this validation.